In [4]:
!conda activate chatgpt
!pip install -r requirements.txt
!pip install python-dotenv

^C


### 4.1 FewShotPromptTemplate

In [18]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler 
from dotenv import load_dotenv

load_dotenv()

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
    )

t = PromptTemplate.from_template("What is the capital of {country}?")
###
# t = PromptTemplate(
#     template='Wat is the capital of {country}?',
#     input_variables=['country'],
# )
###

t.format(country='France') # t == prompt template


'What is the capital of France?'

In [21]:
chat.predict('What do you know about France?')

'France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral. France is also known for its wine production, fashion industry, and art scene. The country has a diverse landscape, including mountains, beaches, and countryside. French is the official language, and the currency is the Euro. France is a member of the European Union and is one of the most visited countries in the world.'

In [24]:
examples = [
    {
        'question': 'What do you know about France?',
        'answer': '''
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        ''',
    },
    {
        'question': 'What do you know about Italy?',
        'answer': '''
        Here is what I know:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        ''',
    },
    {
        'question': 'What do you know about Greece?',
        'answer': '''
        Here is what I know:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        ''',
    },
]

example_template = '''
    Human: {question}
    AI: {answer}
'''

example_prompt = PromptTemplate.from_template(example_template) # param: 'Human: {question}\nAI:{answer}'로 대체 가능.

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    suffix='Human: What do you know about {country}?',
    input_variables=['country']
)

# prompt.format(country='Germany')

chain = prompt | chat
chain.invoke({
    'country': 'Germany'
})

AI: 
        Here is what I know:
        Capital: Berlin
        Language: German
        Food: Bratwurst and Sauerkraut
        Currency: Euro

AIMessageChunk(content='AI: \n        Here is what I know:\n        Capital: Berlin\n        Language: German\n        Food: Bratwurst and Sauerkraut\n        Currency: Euro')

### 4.2 FewShotChatMessagePromptTemplate

In [26]:


from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler 
from dotenv import load_dotenv


examples = [
    {
        'country': 'France',
        'answer': '''
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        ''',
    },
    {
        'country': 'Italy',
        'answer': '''
        Here is what I know:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        ''',
    },
    {
        'country': 'Greece',
        'answer': '''
        Here is what I know:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        ''',
    },
]

example_template = '''
    Human: {question}
    AI: {answer}
'''

example_prompt = ChatPromptTemplate.from_messages([
    ('human', 'What do you know about {country}?'),
    ('ai', '{answer}')
]) # param: 'Human: {question}\nAI:{answer}'로 대체 가능.

# 예제를 형식화
example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a geography expert. You give short answers.'),
    example_prompt,
    ('human', 'What do you know about {country}?')
])

chain = final_prompt | chat
chain.invoke({
    'country': 'Germany'
})


        Here is what I know:
        Capital: Berlin
        Language: German
        Food: Bratwurst and Sauerkraut
        Currency: Euro
        

AIMessageChunk(content='\n        Here is what I know:\n        Capital: Berlin\n        Language: German\n        Food: Bratwurst and Sauerkraut\n        Currency: Euro\n        ')

### 4.3 LengthBasedExampleSelector

In [52]:

from langchain.chat_models import ChatOpenAI
from langchain.prompts import example_selector
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts.prompt import PromptTemplate
# from langchain.prompts.example_selector import LengthBasedExampleSelector
from langchain.prompts.example_selector.base import BaseExampleSelector

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)


examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]


class RandomExampleSelector(BaseExampleSelector):
    def __init__(self, examples):
        self.examples = examples

    def add_example(self, example):
        self.examples.append(example)

    def select_examples(self, input_variables):
        from random import choice

        return [choice(self.examples)]


example_prompt = PromptTemplate.from_template("Human: {question}\nAI: {answer}")

# example_selector = LengthBasedExampleSelector(
#     examples=examples,
#     example_prompt=example_prompt,
#     max_length=150,   # few shot learning에 사용할 예제의 양
# )

example_selector = RandomExampleSelector(
    examples=examples,
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"],
)

prompt.format(country="Brazil")
# 'Human: What do you know about Italy?\nAI:\n        I know this:\n        Capital: Rome\n        Language: Italian\n        Food: Pizza and Pasta\n        Currency: Euro\n        \n\nHuman: What do you know about Brazil?'

'Human: What do you know about Italy?\nAI: \n        I know this:\n        Capital: Rome\n        Language: Italian\n        Food: Pizza and Pasta\n        Currency: Euro\n        \n\nHuman: What do you know about Brazil?'

### 4.4 Serialization and Composition

In [59]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import load_prompt

from langchain.prompts import PromptTemplate
from langchain.prompts.pipeline import PipelinePromptTemplate



prompt = load_prompt('./prompt.yaml')

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

prompt.format(country='xxx')

intro = PromptTemplate.from_template(
    """
    You are a role playing assistant.
    And you are impersonating a {character}
"""
)

example = PromptTemplate.from_template(
    """
    This is an example of how you talk:

    Human: {example_question}
    You: {example_answer}
"""
)

start = PromptTemplate.from_template(
    """
    Start now!

    Human: {question}
    You:
"""
)

final = PromptTemplate.from_template(
    """
    {intro}
                                     
    {example}
                              
    {start}
"""
)

prompts = [
    ("intro", intro),
    ("example", example),
    ("start", start),
]


full_prompt = PipelinePromptTemplate(
    final_prompt=final,
    pipeline_prompts=prompts,
)

# full_prompt.format(
#     character='Pirate',
#     example_question='What is your location?',
#     example_answer='Arrrg! That is a secret!! Arg Arg!!',
#     question='What is your fav food?',
# )

chain = full_prompt | chat

chain.invoke(
    {
        "character": "Pirate",
        "example_question": "What is your location?",
        "example_answer": "Arrrrg! That is a secret!! Arg arg!!",
        "question": "What is your fav food?",
    }
)
 

Arrrrg matey! Me favorite grub be a hearty plate o' salted fish and hardtack! Aye, it be a meal fit for a pirate like meself! Arrrrg!

AIMessageChunk(content="Arrrrg matey! Me favorite grub be a hearty plate o' salted fish and hardtack! Aye, it be a meal fit for a pirate like meself! Arrrrg!")

In [15]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler 
from dotenv import load_dotenv
import ast
import pandas as pd

load_dotenv()

chat = ChatOpenAI(
    model_name='gpt-4-turbo-preview',
    temperature=0,
    streaming=True,
    # callbacks=[
    #     StreamingStdOutCallbackHandler(),
    # ],
    )

examples1 = [
    {
        'text': '발진이안나네요',
        'subject_keywords': '''['발진']'''
        
    },
    {
        'text': '이상품은 발진이 안나요. 좋아요',
        'subject_keywords': '''['상품', '발진']'''
    },
    {
        'text': '사이즈가 너무 커요.',
        'subject_keywords': '''['사이즈']'''
    },
    {
        'text': '이번 제품은 괜찮고, 깨끗해요.',
        'subject_keywords': '''['제품']'''
    },
    {
        'text': '리뷰 싫어!!',
        'subject_keywords': '''['-']'''
    },
]

# example_template = '''
#     Human: {question}
#     AI: {answer}
# '''

example1_prompt = ChatPromptTemplate.from_messages([
    ('human', '{text}'),
    ('ai', '{subject_keywords}')
]) # param: 'Human: {question}\nAI:{answer}'로 대체 가능.

# 예제를 형식화
example1_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example1_prompt,
    examples=examples1
)

stage1_prompt = ChatPromptTemplate.from_messages([
    ('system', """you're data scientist. From now on, you'll find all nouns in the text human says in sequential order. 
            Do not modify the found noun texts in the text.
            If you cannot find any subject keywords, just return '-' as a type of list. 
            You give short answers."""),
    example1_prompt,
    ('human', '{text}')
])

##################################################################################################################################

examples2 = [
    {
        'human': '''
            text: 발진이안나네요
            keywords: ['발진']
        ''',
        'answer': '''{'발진': [('안나네요', 'positive')]}'''
        
    },
    {
        'human': '''
            text: 이상품은 발진이 안나요. 좋아요
            keywords: ['상품', '발진']
        ''',
        'answer': '''{'상품': [('좋아요', 'positive')], '발진': [('안나요', 'positive')]}'''
        
    },
    {
        'human': '''
            text: 사이즈가 너무 커요.
            keywords: ['사이즈']
        ''',
        'answer': '''{'사이즈': [('너무 커요', 'negative')]}'''
        
    },
    {
        'human': '''
            text: 이번 제품은 괜찮고, 깨끗해요.
            keywords: ['제품']
        ''',
        'answer': '''{'제품': [('괜찮고', 'positive'), ('깨끗해요', 'positive')]}'''
        
    },
     {
        'human': '''
            text: 리뷰 싫어!!
            keywords: ['-']
        ''',
        'answer': '''{'-': [('-', '-')]}'''
        
    },
]

# example_template = '''
#     Human: {question}
#     AI: {answer}
# '''

example2_prompt = ChatPromptTemplate.from_messages([
    ('human', '{human}'),
    ('ai', '{answer}')
]) # param: 'Human: {question}\nAI:{answer}'로 대체 가능.

# 예제를 형식화
example2_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example2_prompt,
    examples=examples2
)

stage2_prompt = ChatPromptTemplate.from_messages([
    ('system', """A review text and the subject keywords it contains will be given. 
            You'll find all sentiment keywords corresponding with the given subject keywords in the text. 
            Find all sentiment keywords and tag a sentiment(positive/negative/neutral) as a form of tuple for each keywords given, and put together them as a list.
            After finding all sentimental keywords for every given keywords, put together the results as a form of Dict().
            If you cannot find any words and sentiments for each word, you can use '-' instead.
            DO NOT modify the found words in the text.
            Keep the output format throughly."""),
    example2_prompt,
    ('human', '{human}')
])

##################################################################################################################################

samples = [
    '너무 좋아요~',
    '이 제품 너무 좋아요!!',
    '아이에게 잘맞아서 자주 주문했는데… 잠깐 사이에 기저귀값이 두배?정도 뛴듯해요.너무 급격한 가격변화라 다른 기저귀를 찾아볼까 고민중입니다.',
    '밴드가 부드러워서 좋아요샘방지포켓이 없어서 가끔 앞으로 샐때가있기는 했어요 그래도 재구매의사는 있어요',
    '출산 내년인데 네이버 멤버쉽 행사도 있고 세일도 있고!! 미리 구매했어요~ 많으면 많을수록 좋다고 하고 쌍둥이 출산예정이라 이것저것 손수건 종류별로 많이 구매했어요기저귀는 목욕타월, 낮잠 이불 등등 다용도로 사려구 구매했어요~ 저는 디자인이 있는게 확실히 이쁘네요~ 엄마취향으로다 골랐슴다 ㅎㅎ',
    '애기엄마들 사이에서 하기스가 잘샌다고 하던데 잘 몸에 맞춰서 밴딩하면 괜찮은것 같아요~두께는 정말 얇아요~흡수력도 괜찮고 통풍이 잘 될것 같아요쓰고 아직까진 발진은 안보이네요^^처음쓰는거라서 한팩 써보고 더사용할지 고민하려고 해요~',
    '배송 빠르고 좋네요물놀이 하려고 샀어요 ㅎㅎ 다른데 보다 저렴했어요입혀봤는데 진짜 신기하네요 ㅋㅋ 보기에는 일반 기저귀랑 별로 다른게 없어 보였눈데 방수력 !!ㅋㅋ 일반 기저귀보다 얇아요요긴하게 잘 썼습니다 ㅎ'
]

st1_chain = stage1_prompt | chat
st2_chain = stage2_prompt | chat

final_df = pd.DataFrame(columns=['text_no', 'subject_word', 'sentiment_word', 'sentiment_val', 'texts'])
for text_idx, sample in enumerate(samples, start=1):
    st1_answer = st1_chain.invoke({
        'text': sample
    })
    # print(st1_answer.content)
    st1_list = st1_answer.content
    # print(st1_list)

    st2_answer = st2_chain.invoke({
        'human': 'text: {}\nkeywords: {}'.format(sample, st1_list)
    })
    print(st2_answer.content)

    st2_dict = ast.literal_eval(st2_answer.content)
    for key, val_list in st2_dict.items():
        size = len(val_list)
        temp_dict = {'text_no': [text_idx] * size, 'subject_word': [key] * size, 'sentiment_word': [x[0] for x in val_list], 'sentiment_val': [x[1] for x in val_list], 'texts': [sample] * size}
        temp_df = pd.DataFrame.from_dict(temp_dict)
        final_df = pd.concat([final_df, temp_df], axis=0)

    final_df.to_csv('chatgpt4.0_sample_result.csv', encoding='utf-8-sig', index=False)


    

{'-': [('-', '-')]}
{'제품': [('너무 좋아요', 'positive')]}
{'아이': [('잘맞아서', 'positive')], '사이': [('잠깐', 'neutral')], '기저귀값': [('두배?정도 뛴듯해요', 'negative')], '가격변화': [('급격한', 'negative')], '기저귀': [('다른 기저귀를 찾아볼까 고민중입니다', 'negative')]}
{'밴드': [('부드러워서', 'positive')], '샘방지포켓': [('없어서', 'negative'), ('샐때가있기는 했어요', 'negative')], '재구매의사': [('있어요', 'positive')]}
{'출산': [('미리 구매했어요', 'positive'), ('쌍둥이 출산예정이라', 'positive')], '내년': [('-', '-')], '네이버': [('멤버쉽 행사도 있고', 'positive')], '멤버쉽': [('행사도 있고', 'positive')], '행사': [('있고', 'positive')], '세일': [('있고', 'positive')], '쌍둥이': [('출산예정이라', 'positive')], '출산예정': [('쌍둥이 출산예정이라', 'positive')], '손수건': [('많이 구매했어요', 'positive')], '종류': [('종류별로 많이 구매했어요', 'positive')], '기저귀': [('구매했어요', 'positive')], '목욕타월': [('목욕타월,', 'positive')], '낮잠': [('낮잠 이불', 'positive')], '이불': [('낮잠 이불', 'positive')], '디자인': [('디자인이 있는게 확실히 이쁘네요', 'positive')], '엄마': [('엄마취향으로다 골랐슴다', 'positive')], '취향': [('엄마취향으로다 골랐슴다', 'positive')]}
{'애기엄마들': [('-', '-')], '사이': [('-', '-')], '하기스'

In [7]:
##### 1Q Ver. TEST #####

from langchain.chat_models import ChatOpenAI
from langchain.prompts.pipeline import PipelinePromptTemplate
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.prompt import PromptTemplate
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler, get_openai_callback 
# from langchain.globals import set_llm_cache, set_debug
# from langchain.cache import InMemoryCache
from dotenv import load_dotenv
import ast
import pandas as pd

load_dotenv()


# set_llm_cache(InMemoryCache())
# set_debug(True)

chat = ChatOpenAI(
    model_name='gpt-4-turbo-preview',
    temperature=0,
    streaming=True,
    # callbacks=[
    #     StreamingStdOutCallbackHandler(),
    # ],
    )

examples = [
    {
        'text': '발진이안나네요',
        'answer': '''{"발진": [("안나네요", "positive")]}'''
        
    },
    {
        'text': '이상품은 발진이 안나요. 좋아요',
        'answer': '''{"상품": [("좋아요", "positive")], "발진": [("안나요", "positive")]}'''
        
    },
    {
        'text': '사이즈가 너무 커요.',
        'answer': '''{"사이즈": [("너무 커요", "negative")]}'''
    },
    {
        'text': '이번 제품은 괜찮고, 깨끗해요.',
        'answer': '''{"제품": [("괜찮고", "positive"), ("깨끗해요", "positive")]}'''
    },
    {
        'text': '리뷰 싫어!!',
        'answer': '''{"-": [("-", "-")]}'''
    },
]

intro = PromptTemplate.from_template(
    """
    You're data labeller for NER Model. At first, you'll find all nouns in the text human says in sequential order. 
    Do not modify the found noun texts in the text.
    I'm going to call them SUBJECT KEYWORDS.
    If you cannot find any subject keywords, just return '-' as a type of list. 
   
    Then, you'll find all sentiment keywords and tag a sentiment(positive/negative/neutral) for each subject keywords.
    If you cannot find any words and sentiments for each word, you can use '-' instead.
    Again, DO NOT modify the found words in the text.

    Final output format will be string of Dict() type, which has subject keywords as keys, 
    and a list of tuples (sentiment keywords, sentiment) as its values.
    Keep the output format below throughly.
"""
)

example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{text}'),
    ('ai', '{answer}')
])

example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

start = PromptTemplate.from_template(
    """
    Start now!

    Human: {question}
    You:
"""
)

final = PromptTemplate.from_template(
    """
    {intro}
                                     
    {examples}

    {start}
"""
)

prompts = [
    ("intro", intro),
    ("examples", example_prompt),
    ("start", start),
]


full_prompt = PipelinePromptTemplate(
    final_prompt=final,
    pipeline_prompts=prompts,
)

# full_prompt.format(
#     character='Pirate',
#     example_question='What is your location?',
#     example_answer='Arrrg! That is a secret!! Arg Arg!!',
#     question='What is your fav food?',
# )

chain = full_prompt | chat

samples = [
    '너무 좋아요~',
    '이 제품 너무 좋아요!!',
    '아이에게 잘맞아서 자주 주문했는데… 잠깐 사이에 기저귀값이 두배?정도 뛴듯해요.너무 급격한 가격변화라 다른 기저귀를 찾아볼까 고민중입니다.',
    '밴드가 부드러워서 좋아요샘방지포켓이 없어서 가끔 앞으로 샐때가있기는 했어요 그래도 재구매의사는 있어요',
    '출산 내년인데 네이버 멤버쉽 행사도 있고 세일도 있고!! 미리 구매했어요~ 많으면 많을수록 좋다고 하고 쌍둥이 출산예정이라 이것저것 손수건 종류별로 많이 구매했어요기저귀는 목욕타월, 낮잠 이불 등등 다용도로 사려구 구매했어요~ 저는 디자인이 있는게 확실히 이쁘네요~ 엄마취향으로다 골랐슴다 ㅎㅎ',
    '애기엄마들 사이에서 하기스가 잘샌다고 하던데 잘 몸에 맞춰서 밴딩하면 괜찮은것 같아요~두께는 정말 얇아요~흡수력도 괜찮고 통풍이 잘 될것 같아요쓰고 아직까진 발진은 안보이네요^^처음쓰는거라서 한팩 써보고 더사용할지 고민하려고 해요~',
    '배송 빠르고 좋네요물놀이 하려고 샀어요 ㅎㅎ 다른데 보다 저렴했어요입혀봤는데 진짜 신기하네요 ㅋㅋ 보기에는 일반 기저귀랑 별로 다른게 없어 보였눈데 방수력 !!ㅋㅋ 일반 기저귀보다 얇아요요긴하게 잘 썼습니다 ㅎ'
]

# with get_openai_callback() as usage:
final_df = pd.DataFrame(columns=['text_no', 'subject_word', 'sentiment_word', 'sentiment_val', 'texts'])
for text_idx, sample in enumerate(samples, start=1):
    answer = chain.invoke(
        {
            "question": sample
        }
    )
    print(answer.content)

    rslt_dict = ast.literal_eval(answer.content)
    for key, val_list in rslt_dict.items():
        size = len(val_list)
        temp_dict = {'text_no': [text_idx] * size, 'subject_word': [key] * size, 'sentiment_word': [x[0] for x in val_list], 'sentiment_val': [x[1] for x in val_list], 'texts': [sample] * size}
        temp_df = pd.DataFrame.from_dict(temp_dict)
        final_df = pd.concat([final_df, temp_df], axis=0)

final_df.to_csv('chatgpt4.0_sample_result_v2.csv', encoding='utf-8-sig', index=False)
        # print(usage)

{"-": [("너무 좋아요~", "positive")]}
{"제품": [("너무 좋아요", "positive")]}
{"기저귀값": [("두배", "negative"), ("급격한", "negative"), ("가격변화", "negative")], "기저귀": [("찾아볼까", "neutral")]}
{"밴드": [("부드러워서", "positive"), ("좋아요", "positive")], "샘방지포켓": [("없어서", "negative")], "재구매의사": [("있어요", "positive")]}
{"출산": [("내년인데", "neutral"), ("쌍둥이 출산예정이라", "neutral")], "멤버쉽": [("네이버", "neutral")], "행사": [("있고", "positive")], "세일": [("있고", "positive")], "손수건": [("이것저것", "neutral"), ("종류별로", "neutral")], "기저귀": [("목욕타월,", "neutral"), ("낮잠 이불", "neutral")], "디자인": [("확실히 이쁘네요", "positive")], "엄마": [("엄마취향으로다", "positive")]}
{"하기스": [("잘샌다고", "positive"), ("괜찮은것", "positive")], "밴딩": [("괜찮은것", "positive")], "두께": [("얇아요", "positive")], "흡수력": [("괜찮고", "positive")], "통풍": [("잘 될것", "positive")], "발진": [("안보이네요", "positive")], "팩": [("-", "-")]}
{"배송": [("빠르고", "positive"), ("좋네요", "positive")], "물놀이": [("-", "-")], "저렴했어요": [("저렴했어요", "positive")], "기저귀": [("일반", "neutral"), ("별로 다른게 없어 보였눈데", "neutral"), ("방수력", "pos

In [8]:
##### 1Q Ver-2. TEST #####

from langchain.chat_models import ChatOpenAI
from langchain.prompts.pipeline import PipelinePromptTemplate
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.prompt import PromptTemplate
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler, get_openai_callback 
# from langchain.globals import set_llm_cache, set_debug
# from langchain.cache import InMemoryCache
from dotenv import load_dotenv
import ast
import pandas as pd
import re
import os

from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough


cache_dir = LocalFileStore("./.cache/")

load_dotenv()

# set_llm_cache(InMemoryCache())
# set_debug(True)

chat = ChatOpenAI(
        model_name='gpt-4-turbo-preview',
        temperature=0,
        streaming=True,
        # callbacks=[
        #     StreamingStdOutCallbackHandler(),
        # ],
    )

instruction = PromptTemplate.from_template(
    """
    You're a data labeller for NER Model. Here is what you should do.

    Each paragraph of the following context starts with index. (e.g. [1], [2], [3]...)
    You'll find subject keywords and its sentiments for each paragraph according to the instruction below.

    At first, you'll find all nouns in sequential order throughout a paragraph. 
    Do not modify the found noun texts in the text.
    I'm going to call them SUBJECT KEYWORDS.
    If you cannot find any subject keywords, just return '-'. 
   
    Then, you'll find all sentiment keywords and tag a sentiment(positive/negative/neutral) for each subject keywords in the document.
    If the subject keyword is '-', find all sentiment keywords and tag a sentiment(positive/negative/neutral) through the document.
    If you cannot find any words and sentiments for each word, you can use '-' instead.
    Again, DO NOT modify the found words in the text.

    Output format will be like the example below.
    Keep the output format below throughly.
"""
)

examples = ChatPromptTemplate.from_messages([
    ('human', '''Example Context: 
                [0] 발진이안나네요 [1] 이상품은 발진이 안나요. 좋아요 [2] 사이즈가 너무 커요. [3] 이번 제품은 괜찮고, 깨끗해요. [4] 리뷰 싫어!!'''),
    ('ai', '''
            0: (발진, 안나네요, positive)
            1: (상품, 좋아요, positive), (발진, 안나요, positive)
            2: (사이즈, 너무 커요, negative)
            3: (제품, 괜찮고, positive), (제품, 깨끗해요, positive)
            4: (-, -, -)
        ''')
])

# examples = PromptTemplate.from_template(
#     '''
#     This is an example of context and what you should return:

#     Example Context: 
#     [0] 발진이안나네요 [1] 이상품은 발진이 안나요. 좋아요 [2] 사이즈가 너무 커요. [3] 이번 제품은 괜찮고, 깨끗해요. [4] 리뷰 싫어!!

#     You: 

# '''
# )
# [{"발진": [("안나네요", "positive")]}, {"상품": [("좋아요", "positive")], "발진": [("안나요", "positive")]}, {"사이즈": [("너무 커요", "negative")]}, {"제품": [("괜찮고", "positive"), ("깨끗해요", "positive")]}, {"-": [("-", "-")]}]


samples = [
    '너무 좋아요~',
    '이 제품 너무 좋아요!!',
    '아이에게 잘맞아서 자주 주문했는데… \n\n잠깐 사이에 기저귀값이 두배?정도 뛴듯해요.너무 급격한 가격변화라 다른 기저귀를 찾아볼까 고민중입니다.',
    '밴드가 부드러워서 좋아요샘방지포켓이 없어서 가끔 앞으로 샐때가있기는 했어요 그래도 재구매의사는 있어요',
    '출산 내년인데 네이버 멤버쉽 행사도 있고 세일도 있고!! 미리 구매했어요~ 많으면 많을수록 좋다고 하고 쌍둥이 출산예정이라 이것저것 손수건 종류별로 많이 구매했어요기저귀는 목욕타월, 낮잠 이불 등등 다용도로 사려구 구매했어요~ 저는 디자인이 있는게 확실히 이쁘네요~ 엄마취향으로다 골랐슴다 ㅎㅎ',
    '애기엄마들 사이에서 하기스가 잘샌다고 하던데 잘 몸에 맞춰서 밴딩하면 괜찮은것 같아요~두께는 정말 얇아요~흡수력도 괜찮고 통풍이 잘 될것 같아요쓰고 아직까진 발진은 안보이네요^^처음쓰는거라서 한팩 써보고 더사용할지 고민하려고 해요~',
    '배송 빠르고 좋네요물놀이 하려고 샀어요 ㅎㅎ 다른데 보다 저렴했어요입혀봤는데 진짜 신기하네요 ㅋㅋ 보기에는 일반 기저귀랑 별로 다른게 없어 보였눈데 방수력 !!ㅋㅋ 일반 기저귀보다 얇아요요긴하게 잘 썼습니다 ㅎ'
]

samples = ['[{}] '.format(idx) + re.sub('\n', ' ', sample) for idx, sample in enumerate(samples)]

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator='\n\n',
)

docs = splitter.create_documents(samples)
print(len(docs), docs)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

retriver = vectorstore.as_retriever()

documents = PromptTemplate.from_template(
    """
    Here's the context:\n\n{context}
"""
)

final = PromptTemplate.from_template(
    """
    {instruction}
                                     
    {examples}

    {documents}

    {start}
"""
)

prompts = [
    ("instruction", instruction),
    ("examples", examples),
    ("documents", documents)
]


full_prompt = PipelinePromptTemplate(
    final_prompt=final,
    pipeline_prompts=prompts,
)

chain = (
    {
        "context": retriver,
        "start": RunnablePassthrough(),
    }
    | full_prompt
    | chat
)

response = chain.invoke("Follow the instruction above and return output!")
print(response.content)


7 [Document(page_content='[0] 너무 좋아요~'), Document(page_content='[1] 이 제품 너무 좋아요!!'), Document(page_content='[2] 아이에게 잘맞아서 자주 주문했는데…   잠깐 사이에 기저귀값이 두배?정도 뛴듯해요.너무 급격한 가격변화라 다른 기저귀를 찾아볼까 고민중입니다.'), Document(page_content='[3] 밴드가 부드러워서 좋아요샘방지포켓이 없어서 가끔 앞으로 샐때가있기는 했어요 그래도 재구매의사는 있어요'), Document(page_content='[4] 출산 내년인데 네이버 멤버쉽 행사도 있고 세일도 있고!! 미리 구매했어요~ 많으면 많을수록 좋다고 하고 쌍둥이 출산예정이라 이것저것 손수건 종류별로 많이 구매했어요기저귀는 목욕타월, 낮잠 이불 등등 다용도로 사려구 구매했어요~ 저는 디자인이 있는게 확실히 이쁘네요~ 엄마취향으로다 골랐슴다 ㅎㅎ'), Document(page_content='[5] 애기엄마들 사이에서 하기스가 잘샌다고 하던데 잘 몸에 맞춰서 밴딩하면 괜찮은것 같아요~두께는 정말 얇아요~흡수력도 괜찮고 통풍이 잘 될것 같아요쓰고 아직까진 발진은 안보이네요^^처음쓰는거라서 한팩 써보고 더사용할지 고민하려고 해요~'), Document(page_content='[6] 배송 빠르고 좋네요물놀이 하려고 샀어요 ㅎㅎ 다른데 보다 저렴했어요입혀봤는데 진짜 신기하네요 ㅋㅋ 보기에는 일반 기저귀랑 별로 다른게 없어 보였눈데 방수력 !!ㅋㅋ 일반 기저귀보다 얇아요요긴하게 잘 썼습니다 ㅎ')]


KeyError: '발진'

In [14]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.globals import set_llm_cache, set_debug
from langchain.cache import InMemoryCache, SQLiteCache

set_llm_cache(SQLiteCache("cache.db"))


chat = ChatOpenAI(
    temperature=0.1,
    # streaming=True,
    # callbacks=[
    #     StreamingStdOutCallbackHandler(),
    # ],
)

chat.predict("How do you make italian pasta")

'To make Italian pasta, you will need the following ingredients:\n\n- 2 cups of all-purpose flour\n- 2 large eggs\n- Pinch of salt\n\nHere is a step-by-step guide to making Italian pasta:\n\n1. On a clean work surface, pour the flour and create a well in the center.\n2. Crack the eggs into the well and add a pinch of salt.\n3. Using a fork, gradually mix the eggs into the flour until a dough forms.\n4. Knead the dough for about 10 minutes until it is smooth and elastic.\n5. Wrap the dough in plastic wrap and let it rest for at least 30 minutes.\n6. After resting, roll out the dough using a pasta machine or a rolling pin until it is thin.\n7. Cut the dough into your desired shape, such as fettuccine or spaghetti.\n8. Cook the pasta in a large pot of boiling salted water for 2-3 minutes or until al dente.\n9. Drain the pasta and toss with your favorite sauce or toppings.\n\nEnjoy your homemade Italian pasta!'

In [15]:

chat.predict("How do you make italian pasta")

'To make Italian pasta, you will need the following ingredients:\n\n- 2 cups of all-purpose flour\n- 2 large eggs\n- Pinch of salt\n\nHere is a step-by-step guide to making Italian pasta:\n\n1. On a clean work surface, pour the flour and create a well in the center.\n2. Crack the eggs into the well and add a pinch of salt.\n3. Using a fork, gradually mix the eggs into the flour until a dough forms.\n4. Knead the dough for about 10 minutes until it is smooth and elastic.\n5. Wrap the dough in plastic wrap and let it rest for at least 30 minutes.\n6. After resting, roll out the dough using a pasta machine or a rolling pin until it is thin.\n7. Cut the dough into your desired shape, such as fettuccine or spaghetti.\n8. Cook the pasta in a large pot of boiling salted water for 2-3 minutes or until al dente.\n9. Drain the pasta and toss with your favorite sauce or toppings.\n\nEnjoy your homemade Italian pasta!'

In [16]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import get_openai_callback

chat = ChatOpenAI(
    temperature=0.1,
)

with get_openai_callback() as usage:
    chat.predict('What is the recipe for soju?')
    print(usage)

Tokens Used: 211
	Prompt Tokens: 15
	Completion Tokens: 196
Successful Requests: 1
Total Cost (USD): $0.00041450000000000005


In [17]:
from langchain.chat_models import ChatOpenAI
from langchain.llms.openai import OpenAI

chat = OpenAI(
    temperature=0.1,
    max_tokens=450,
    model='gpt-3.5-turbo-16k'
)

chat.save("model.json")

In [18]:
from langchain.chat_models import ChatOpenAI
from langchain.llms.openai import OpenAI
from langchain.llms.loading import load_llm


chat = load_llm("model.json")

chat

c:\Users\Admin\anaconda3\envs\chatgpt\lib\site-packages\langchain\llms\openai.py:216: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(
c:\Users\Admin\anaconda3\envs\chatgpt\lib\site-packages\langchain\llms\openai.py:811: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(


OpenAIChat(client=<class 'openai.api_resources.chat_completion.ChatCompletion'>, model_name='gpt-3.5-turbo-16k', model_kwargs={'temperature': 0.1, 'max_tokens': 450, 'top_p': 1, 'frequency_penalty': 0, 'presence_penalty': 0, 'n': 1, 'request_timeout': None, 'logit_bias': {}})

# 5. Memory

### Common Function
* save_context()
* load_memory_variables()

### 5 types of memory
* CoversationBufferMemory : 대화 내용 전체 저장. 비효울적. 고비용. 텍스트 자동완성 기능 구현 시 유용.
* ConversationBufferWindowMemory : 대화의 특정 부분만 저장.
* ConversationSummaryMemory : 대화 내용을 요약해 저장.
* ConversationSummaryBufferMemory : buffer window memory + summary memory; limit을 정하고 그 이전 메세지들은 요약해서 기억.
* ConversationKGMemory : "Knowledge Graph"; 

In [23]:
from langchain.memory import ConversationBufferMemory


memory = ConversationBufferMemory(return_messages=True)

memory.save_context({'input': 'Hi!'}, {'output': 'How are you?'})

memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi!'), AIMessage(content='How are you?')]}

In [24]:
memory.save_context({'input': 'Hi!'}, {'output': 'How are you?'})

memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi!'),
  AIMessage(content='How are you?'),
  HumanMessage(content='Hi!'),
  AIMessage(content='How are you?')]}

In [27]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    return_messages=True,
    k=4,   # 버퍼 윈도우의 사이즈, 몇 개의 메시지를 저장할지 지정 
)

def add_message(input, output):
    memory.save_context({'input': input}, {'output': output})


add_message(1, 1)
add_message(2, 2)
add_message(3, 3)
add_message(4, 4)

memory.load_memory_variables({})

{'history': [HumanMessage(content='1'),
  AIMessage(content='1'),
  HumanMessage(content='2'),
  AIMessage(content='2'),
  HumanMessage(content='3'),
  AIMessage(content='3'),
  HumanMessage(content='4'),
  AIMessage(content='4')]}

In [28]:
add_message(5, 5)

memory.load_memory_variables({})

{'history': [HumanMessage(content='2'),
  AIMessage(content='2'),
  HumanMessage(content='3'),
  AIMessage(content='3'),
  HumanMessage(content='4'),
  AIMessage(content='4'),
  HumanMessage(content='5'),
  AIMessage(content='5')]}

In [29]:
from langchain.memory import ConversationSummaryMemory
from langchain.chat_models import ChatOpenAI


llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryMemory(llm=llm)

def add_message(input, output):
    memory.save_context({'inputs': input}, {'output': output})


def get_history():
    return memory.load_memory_variables({})


add_message('Hi I\'m Nicolas, I live in South Korea', 'Wow that is so cool!')

In [30]:
add_message('South Korea is so pretty', 'I wish I could visit there')

In [31]:
get_history()

{'history': 'Nicolas introduces himself as living in South Korea. The AI responds by expressing admiration for his location, wishing it could visit there because South Korea is so pretty.'}

In [38]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI


llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=50,
    return_messages=True
)

def add_message(input, output):
    memory.save_context({'inputs': input}, {'output': output})


def get_history():
    return memory.load_memory_variables({})


add_message('Hi I\'m Nicolas, I live in South Korea', 'Wow that is so cool!')
add_message('South Korea is so pretty', 'I wish I could visit there')

get_history()

{'history': [HumanMessage(content="Hi I'm Nicolas, I live in South Korea"),
  AIMessage(content='Wow that is so cool!'),
  HumanMessage(content='South Korea is so pretty'),
  AIMessage(content='I wish I could visit there')]}

In [39]:
add_message('How far is Korea from Argentina?', 'I don\'t know! Super far!')

get_history()

{'history': [SystemMessage(content='Nicolas introduces himself as living in South Korea. The AI responds by expressing admiration for his location.'),
  HumanMessage(content='South Korea is so pretty'),
  AIMessage(content='I wish I could visit there'),
  HumanMessage(content='How far is Korea from Argentina?'),
  AIMessage(content="I don't know! Super far!")]}

In [41]:
from langchain.memory import ConversationKGMemory
from langchain.chat_models import ChatOpenAI


llm = ChatOpenAI(temperature=0.1)

memory = ConversationKGMemory(
    llm=llm,
    return_messages=True
)

def add_message(input, output):
    memory.save_context({'inputs': input}, {'output': output})


def get_history():
    return memory.load_memory_variables({})


add_message('Hi I\'m Nicolas, I live in South Korea', 'Wow that is so cool!')

In [42]:
memory.load_memory_variables({'inputs': 'Who is Nicolas?'})

{'history': [SystemMessage(content='On Nicolas: Nicolas lives in South Korea.')]}

In [46]:
add_message('Nicolas likes Kimchi.', 'Wow that is so cool!')
memory.load_memory_variables({'inputs': 'What does Nicolas like?'})

{'history': [SystemMessage(content='On Nicolas: Nicolas lives in South Korea. Nicolas likes Kimchi.')]}

### 5.5 Memory on LLMChain

In [47]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history",
)

template = """
    You are a helpful AI talking to a human.

    {chat_history}
    Human:{question}
    You:
"""

### off-the-shelf: chain for general use   <->   custo

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=PromptTemplate.from_template(template),
    verbose=True,
)

chain.predict(question="My name is Nico")



> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    
    Human:My name is Nico
    You:


> Finished chain.


'Hello Nico! How can I assist you today?'

In [48]:
chain.predict(question="I live in Seoul")



> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    Human: My name is Nico
AI: Hello Nico! How can I assist you today?
    Human:I live in Seoul
    You:


> Finished chain.


"That's great to know! How can I assist you with information or tasks related to Seoul?"

In [49]:
chain.predict(question="What is my name?")



> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    Human: My name is Nico
AI: Hello Nico! How can I assist you today?
Human: I live in Seoul
AI: That's great to know! How can I assist you with information or tasks related to Seoul?
    Human:What is my name?
    You:


> Finished chain.


'Your name is Nico.'